### 1. Mount Google Drive
This cell mounts your Google Drive to access files and save results directly from the Colab environment.

In [4]:
from google.colab import drive
# Mount Google Drive to access files and save results
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Download Dataset from Roboflow
This cell sets up the working directory in Google Drive and downloads the specified dataset from Roboflow using its API. Replace `kYs18KKtZGBRh6bRrxXJ` with your actual Roboflow API key if you are using your own project.

In [5]:
import os
from roboflow import Roboflow

# Create and change working directory to your Drive folder
os.makedirs("/content/drive/MyDrive/TrafficDetectPK", exist_ok=True)
os.chdir("/content/drive/MyDrive/TrafficDetectPK")

# Download dataset from Roboflow
try:
    # Initialize Roboflow with your API key
    rf = Roboflow(api_key="kYs18KKtZGBRh6bRrxXJ")
    # Select the workspace and project
    project = rf.workspace("moiz-chauhan-u4zyj").project("traffic-erawl")
    # Download a specific version of the dataset in YOLOv8 format
    dataset = project.version(6).download("yolov8")

    print("Dataset downloaded successfully!")
    print(f"Dataset location: {dataset.location}")

except Exception as e:
    print("Error occured: ", e)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to traffic-6 in yolov8:: 100%|██████████| 2715/2715 [00:28<00:00, 94.70it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset downloaded successfully!
Dataset location: /content/drive/MyDrive/TrafficDetectPK/traffic-6


### 3. Train YOLOv8 Model
This cell initializes a YOLOv8n model and trains it using the downloaded dataset. The training parameters such as epochs, image size, batch size, and project/name for saving results are configured here.

In [7]:
from ultralytics import YOLO

# Load a pretrained YOLOv8n model
model = YOLO("yolov8n.pt")

# Train the model with specified parameters
results = model.train(
    data="/content/drive/MyDrive/TrafficDetectPK/traffic-6/data.yaml", # Path to the dataset YAML file
    epochs=50,      # Number of training epochs
    imgsz=640,      # Image size for training
    batch=16,       # Batch size
    name="TrafficDetectPK", # Name for saving the training run
    project="/content/drive/MyDrive/TrafficDetectPK/runs", # Directory to save project results
    patience=15,    # Number of epochs to wait for improvement before early stopping
    workers=2,      # Number of worker threads for data loading
    verbose=True    # Print detailed output during training
)

print("Training complete!")

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/TrafficDetectPK/traffic-6/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=TrafficDetectPK-2, nbs=64, nms=False, opset=None, optimize=False, optimiz

### 4. Validate Trained Model
This cell loads the best-trained model weights and performs validation on the dataset. It then prints key metrics like mAP50, mAP50-95, Precision, and Recall to evaluate the model's performance.

In [10]:
from ultralytics import YOLO

# Load the best-trained model weights from the specified path
model = YOLO("/content/drive/MyDrive/TrafficDetectPK/runs/TrafficDetectPK-2/weights/best.pt")

# Perform validation on the dataset using the loaded model
metrics = model.val(
    data="/content/drive/MyDrive/TrafficDetectPK/traffic-6/data.yaml" # Path to the dataset YAML file for validation
)

# Print key performance metrics
print(f"mAP50: {metrics.box.map50:.3f}")     # Mean Average Precision at IoU=0.5
print(f"mAP50-95: {metrics.box.map:.3f}")  # Mean Average Precision across IoU thresholds from 0.5 to 0.95
print(f"Precision: {metrics.box.p.mean():.3f}")  # Average Precision
print(f"Recall: {metrics.box.r.mean():.3f}")     # Average Recall

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.5 ms, read: 9.7±5.4 MB/s, size: 18.1 KB)
val: Scanning /content/drive/MyDrive/TrafficDetectPK/traffic-6/valid/labels.cache... 170 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 170/170 59.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.5it/s 7.3s
                   all        170       2424      0.737      0.748      0.787      0.588
         Auto Rickshaw        133        313       0.84      0.962      0.949      0.784
   Car-Jeeps-vans-Taxi        170       1145      0.852      0.865      0.921       0.74
         Cart-Chingchi          7          9      0.592      0.487      0.588      0.348
           Large Buses         47         53      0.848      0.811      0.891      0.719
            Mini Buses